<a href="https://colab.research.google.com/github/Dacon-TeamPK/dacon-physics-vision-ai/blob/model/swin/swin_0.091.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 라이브러리 및 기초 설정

In [19]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import swin_t, Swin_T_Weights
from tqdm.auto import tqdm

In [28]:
# 하이퍼파라미터 설정
CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 30,
    'PATIENCE': 10,  # 성능이 몇 번 이상 안 오르면 멈출지
    'LEARNING_RATE': 3e-5,  # 1e-4, 3e-5 등 실험 해보기
    'BATCH_SIZE': 8,
    'SEED': 42
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

os.chdir('/content/drive/MyDrive/Colab_Notebooks/Dacon_PhysicsAI')
seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


## 데이터 로드

In [29]:
train_df = pd.read_csv('./train.csv')
val_df = pd.read_csv('./dev.csv')

train_df['split'] = 'train'
val_df['split'] = 'dev'
all_df = pd.concat([train_df, val_df], ignore_index=True)

## 데이터셋 클래스 정의 및 데이터 로더

In [30]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        split = self.df.iloc[idx].get('split', 'train')  # split 컬럼 읽기

        if isinstance(self.root_dir, dict):
            folder_path = os.path.join(self.root_dir[split], sample_id)
        else:
            folder_path = os.path.join(self.root_dir, sample_id)

        # 2개 뷰 로드
        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        # 테스트(추론) 모드일 경우 이미지 리스트만 반환
        if self.is_test:
            return views

        # 학습/검증 모드일 경우 라벨 함께 반환
        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

In [31]:
train_transform = transforms.Compose([
    ## Your Augmentation
    # transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomResizedCrop(CFG['IMG_SIZE'], scale=(0.6,1.0)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.4,
        hue=0.15
    ),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),  # 추가

    transforms.ToTensor(),

    transforms.RandomErasing(p=0.3),

    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 1. 학습/검증 세트 준비 (is_test=False 설정)
train_dataset = MultiViewDataset(
    all_df,
    root_dir={'train': './train', 'dev': './dev'},  # 딕셔너리로 전달
    transform=train_transform,
    is_test=False
)
val_dataset = MultiViewDataset(val_df, './dev', test_transform, is_test=False)

train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

# 2. 테스트 세트 준비 (is_test=True 설정)
test_df = pd.read_csv('./sample_submission.csv')
test_dataset = MultiViewDataset(test_df, './test', test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

## 모델 정의(MultiView)

In [32]:
class MultiViewSwin(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.encoder = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)
        self.encoder.head = nn.Identity()
        feat_dim = 768  # swin_t output

        # 두 뷰 간 상호작용을 학습하는 cross-attention
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=feat_dim, num_heads=8, batch_first=True, dropout=0.1
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim * 2),
            nn.Linear(feat_dim * 2, 256),
            nn.GELU(),
            nn.Dropout(0.4),        # 0.2 → 0.4 (과적합 억제)
            nn.Linear(256, 1)
        )

    def forward(self, views):
        f1 = self.encoder(views[0])  # (B, 768)
        f2 = self.encoder(views[1])  # (B, 768)

        # cross-attention: f1이 f2를 참조
        f1_seq = f1.unsqueeze(1)  # (B, 1, 768)
        f2_seq = f2.unsqueeze(1)
        attn_out, _ = self.cross_attn(f1_seq, f2_seq, f2_seq)
        attn_feat = attn_out.squeeze(1)  # (B, 768)

        combined = torch.cat((attn_feat, f2), dim=1)  # (B, 1536)
        return self.classifier(combined)

## 학습 및 검증 루프 구현

In [33]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views).view(-1)
            # 1. 시그모이드를 통과시켜 확률값(unstable일 확률)으로 변환
            probs = torch.sigmoid(outputs)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)

    # 대회 공식 LOGLOSS 계산
    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    # Binary Log Loss 공식 직접 적용
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))

    # Accuracy 계산
    acc_score = np.mean((all_probs > 0.5) == all_labels)

    return logloss_score, acc_score

In [34]:
model = MultiViewSwin().to(device)

criterion = nn.BCEWithLogitsLoss()
# CrossEntropyLoss()도 해보기

optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG['LEARNING_RATE'],
    weight_decay=5e-2  # 1e-4 → 5e-2로 강화
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG['EPOCHS'],
    eta_min=1e-7
)

best_logloss = float('inf')
patience_counter = 0

# --- Main Loop ---
for epoch in range(1, CFG['EPOCHS'] + 1):
    avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_logloss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"Epoch [{epoch}]")
    print(f"  - Train Loss: {avg_train_loss:.4f}")
    print(f"  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}")

    # best model 저장
    if val_logloss < best_logloss:
        best_logloss = val_logloss
        patience_counter = 0

        torch.save(model.state_dict(), "best_model.pth")
        torch.save(model.state_dict(), f"best_model_epoch{epoch}.pth")
        print(f"  * Best model 저장완료 (Val Log Loss: {best_logloss:.4f})")
        print(f"---------------------------------------------------")

    else:
        patience_counter += 1
        print(f"  - patience: {patience_counter}/{CFG['PATIENCE']}")

    # early stopping
    if patience_counter >= CFG['PATIENCE']:
        print("Early Stopping 실행됨")
        break

Training:   0%|          | 0/138 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch [1]
  - Train Loss: 0.5817
  - Val Log-Loss: 0.488266 | Val Acc: 0.7500
  * Best model 저장완료 (Val Log Loss: 0.4883)
---------------------------------------------------


Training:   0%|          | 0/138 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch [29]
  - Train Loss: 0.0867
  - Val Log-Loss: 0.100711 | Val Acc: 0.9500
  - patience: 4/10


Training:   0%|          | 0/138 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch [30]
  - Train Loss: 0.1083
  - Val Log-Loss: 0.101310 | Val Acc: 0.9500
  - patience: 5/10


## 추론 및 제출 파일 생성

In [35]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
all_probs = []

with torch.no_grad():
    for views in tqdm(test_loader, desc="Inference"):
        views = [v.to(device) for v in views]

        # 모델 출력 (Logit) -> 시그모이드 -> unstable(1)일 확률
        outputs = model(views).view(-1)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)
# 결과 저장 (컬럼  순서 중요)
submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': all_probs,  # unstable일 확률 저장
    'stable_prob': 1.0 - all_probs # stable일 확률 저장
})

submission.to_csv('submission.csv', encoding='UTF-8-sig', index=False)
print("submission.csv 저장 완료.")

Inference:   0%|          | 0/125 [00:00<?, ?it/s]

submission.csv 저장 완료.
